# Does the DD-Sudoku transformer run DD? And where is the hint-density transition?

**Git commit:** `afe6370` (branch `sudoku-dd`).

**Model under test:** `d0128_h04_L12_ff01024_lr1e-03_pre` @ 900 epochs, from Slurm `48513926_1`
(`$SCRATCH/spin_gnn/sudoku_hparam_sweep_2026-10-19_ep900`), `agg_solve_rate=0.767`, 3,994,146 params.
Config: `SudokuReasoningConfig(n_nodes=24, degree=5, n_colors=4)`, removal-path hints, `norm_first=True`.

**Runs analyzed here**

| Slurm | what | seed |
|---|---|---|
| `48581590` | DD removal-path `T(rho)`, `rho_min` vs N (N = 24..996) | 7,000,000 |
| `48581589` | model transition curve by realized rho, 200 instances/bin | 800,000,000 |
| `48579545` | random-hint ensemble training + transition | 12,345 |
| `48580216` | linear probes by layer | 500,000,000 |
| `48582589` | algorithm ID on DD-vs-heuristic disagreements, 400 instances | 600,000,000 |
| `48583439` | SAE, target-L0 32, layers 0/4/8/12 | 0 |
| `48583444` | causal counterfactual + activation patching, 150 cases | 900,000,000 |
| `48579019` | scratchpad-token variant, 900 epochs | 12,345 |

**Headline.** The hint-density transition is real and already visible at N=24: solve_rate falls
0.99 -> 0.44 as DD's deduction depth `T` rises 1 -> 6. But three independent tests agree the model
is *not* executing DD.

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

S = os.environ["SCRATCH"] + "/spin_gnn"
pd.set_option("display.width", 160)

## 1. Where the transition is

Two hint ensembles had to be separated before any of this made sense.

- **`random`**: independent uniform hints at `round(rho*N)` nodes. DD genuinely fails here, but there
  is **no transition at any N** -- see cell 3. This is the ensemble the earlier deck chased.
- **`removal_path`** (what the models train on): start fully hinted, greedily remove hints while DD
  still recovers everything. Resolvable *by construction* at every rho, so rho sets reasoning **depth**,
  not solvability. The transition is the divergence of DD's deduction time `T` as rho approaches the
  greedy floor `rho_min`.

In [2]:
dd = pd.read_csv(f"{S}/sudoku_dd_removal_small_n_2026-10-19/removal_path_T_vs_rho.csv")
floors = dd.groupby("n_nodes")["rho"].min().rename("rho_min")
print("greedy floor rho_min by N (converges, N-independent):")
print(floors.to_string())

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
for n, g in dd.groupby("n_nodes"):
    g = g[g.frac_paths_reaching_rho >= 0.8].sort_values("rho")
    axes[0].plot(g.rho, g.mean_T, marker="o", ms=3, label=f"N={n}")
axes[0].set_xlabel("hint density rho"); axes[0].set_ylabel("DD deduction depth T")
axes[0].invert_xaxis(); axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)
axes[0].set_title("T diverges toward the floor")
axes[1].plot(floors.index, floors.values, marker="s", color="#d62728")
axes[1].set_xscale("log"); axes[1].set_xlabel("N"); axes[1].set_ylabel("rho_min")
axes[1].axhline(0.477, ls=":", color="#888"); axes[1].grid(alpha=0.3)
axes[1].set_title("floor converges to ~0.477 by N=96")
plt.tight_layout()

greedy floor rho_min by N (converges, N-independent):
n_nodes
24     0.41
48     0.41
96     0.41
192    0.41
384    0.41
996    0.41


## 2. The model transition (Slurm 48581589)

Hypothesis as pre-registered in `scripts/slurm/sudoku2_hint_density_transition.sbatch`: *solve_rate falls
steeply as realized_rho approaches `rho_min`, and the fall tracks `dd_mean_T` rather than rho itself.*

`truncated_rate` is reported per bin to rule out the obvious confound -- a longer reasoning trace needs
more tokens, so a fall in solve_rate could just be a starved generation budget.

**The first half of that hypothesis holds; the second half is NOT established by this eval.**
`r(solve_rate, T) = -0.966` against `r(solve_rate, rho) = +0.944` -- near-identical in magnitude, because
in the removal-path ensemble rho and `T` are collinear *by construction* (removing hints is precisely what
lengthens the deduction chain). So the collapse below is solid, but attributing it to reasoning **depth**
rather than to **density** is not something these numbers can do. Separating the two needs instances
matched on one and varied on the other -- bin equal-rho instances by `T` -- which is the next measurement.

In [3]:
tr = pd.read_csv(f"{S}/sudoku_hint_density_transition_2026-10-19/transition.csv")
tr = tr[tr.checkpoint.str.contains("d0128")].drop_duplicates("realized_rho").sort_values("realized_rho")
print(tr[["realized_rho", "dd_mean_T", "solve_rate", "well_formed_rate", "truncated_rate"]].to_string(index=False))
print("\ntruncated_rate max:", tr.truncated_rate.max(), "-> the fall is NOT a generation-budget artifact")
print("Pearson r(solve_rate, dd_mean_T) =", np.corrcoef(tr.solve_rate, tr.dd_mean_T)[0, 1].round(4))
print("Pearson r(solve_rate, rho)       =", np.corrcoef(tr.solve_rate, tr.realized_rho)[0, 1].round(4))

fig, ax = plt.subplots(figsize=(7.5, 4.4))
ax.plot(tr.dd_mean_T, tr.solve_rate, marker="o", color="#1f77b4", lw=2)
ax.set_xlabel("DD deduction depth T (serial reasoning steps)"); ax.set_ylabel("solve_rate")
ax.grid(alpha=0.3); ax.set_title("Model collapses with reasoning DEPTH")
plt.tight_layout()

 realized_rho  dd_mean_T  solve_rate  well_formed_rate  truncated_rate
     0.453611   6.040000    0.500000              1.00             0.0
     0.475000   5.626667    0.540000              1.00             0.0
     0.506111   5.060000    0.600000              1.00             0.0
     0.543333   4.480000    0.626667              1.00             0.0
     0.583611   4.013333    0.680000              1.00             0.0
     0.625000   3.533333    0.753333              1.00             0.0
     0.666667   3.080000    0.793333              1.00             0.0
     0.708333   2.680000    0.860000              1.00             0.0
     0.750000   2.260000    0.920000              1.00             0.0
     0.791667   1.826667    0.933333              1.00             0.0
     0.833333   1.513333    0.966667              1.00             0.0
     0.875000   1.300000    0.973333              1.00             0.0
     0.916667   1.093333    0.980000              1.00             0.0
     0

### The random-hint ensemble, for contrast (Slurm 48579545)

Flat and non-monotone: solve_rate wanders in 0.64-0.78 across rho 0.35-0.95 and is *highest* at both
extremes (at rho<=0.15 almost nothing is deducible, so `UNRESOLVED` is trivially correct). There is no
transition to find here at any N, which is why the earlier "scale N to ~1700 tokens" plan is retired.

In [4]:
rh = pd.read_csv(f"{S}/sudoku_random_hint_transition_2026-10-19/transition.csv")
print(rh[["rho", "solve_rate"]].to_string(index=False))

 rho  solve_rate
0.00      0.2625
0.05      0.9875
0.10      1.0000
0.15      1.0000
0.20      0.9625
0.25      0.9375
0.30      0.9250
0.35      0.7750
0.40      0.7000
0.45      0.7500
0.50      0.7375
0.55      0.6875
0.60      0.7125
0.65      0.7500
0.70      0.6500
0.75      0.7750
0.80      0.6625
0.85      0.6375
0.90      0.4875
0.95      0.6500
1.00      1.0000


## 3. Is the model running DD? Three independent tests

### 3a. Linear probes by layer (Slurm 48580216)

The contrast that matters is `known_mask` (pure bookkeeping -- trackable by copying tokens and
positions) against `frontier_mask` (**the** DD predicate -- requires the graph table joined to the known
set). Layer 0 is the raw token+position embedding and acts as the control.

In [5]:
pr = pd.read_csv(f"{S}/sudoku_probes_2026-10-19/probes.csv")
pr = pr[pr.checkpoint.str.contains("d0128")]
piv = pr.pivot_table(index="layer", columns="target", values="balanced_accuracy")
cols = [c for c in ["known_mask", "frontier_mask", "remaining_mask", "next_node", "next_color"] if c in piv]
print(piv[cols].round(3).to_string())
print("\npeak known_mask   %.3f  (bookkeeping: strongly computed, early)" % piv["known_mask"].max())
print("peak frontier_mask %.3f  (DD predicate: barely above the 0.57 control)" % piv["frontier_mask"].max())

target  known_mask  frontier_mask  remaining_mask  next_node  next_color
layer                                                                   
0            0.593          0.569           0.608      0.533       0.268
1            0.824          0.596           0.657      0.639       0.363
2            0.898          0.623           0.699      0.666       0.430
3            0.912          0.652           0.711      0.688       0.436
4            0.913          0.660           0.720      0.700       0.438
5            0.914          0.663           0.720      0.705       0.432
6            0.910          0.663           0.719      0.711       0.431
7            0.905          0.659           0.717      0.714       0.440
8            0.893          0.657           0.718      0.731       0.539
9            0.878          0.657           0.723      0.740       0.602
10           0.889          0.660           0.730      0.744       0.624
11           0.871          0.665           0.730  

### 3b. Algorithm ID on disagreements (Slurm 48582589)

`next_node` probing at 0.76 looks like the model knows the deducible set, but it is confounded:
`solve_with_dd_trace` emits each round as `sorted(...)`, so node ids **ascend within a round** and a pure
ordering rule matches DD far above chance with no deducibility at all
(`tests/test_sudoku_algorithm_id.py` pins this at >0.3).

So score **only positions where DD and a baseline disagree**, where the model's prediction is a genuine
vote. `most_known_nbrs` is the discriminating middle case: it uses the graph and is close to DD's forcing
condition without equalling it.

In [6]:
frames = []
for d in ("d0128", "d0064"):
    f = f"{S}/sudoku_algorithm_id_2026-10-19/{d}_h04_L12_ff01024_lr1e-03_pre.csv"
    g = pd.read_csv(f).groupby("baseline").agg(
        n=("model_follows_dd", "size"),
        follows_dd=("model_follows_dd", "mean"),
        follows_baseline=("model_follows_baseline", "mean"))
    g["model"] = d
    frames.append(g.reset_index())
alg = pd.concat(frames)
print(alg.round(3).to_string(index=False))
print("\nAgainst `ascending_after` -- the one baseline that exploits the emission convention --")
print("the model prefers the HEURISTIC over DD, and more so at d_model=64.")

        baseline    n  follows_dd  follows_baseline model
       ascending 2154       0.478             0.144 d0128
 ascending_after 1461       0.334             0.361 d0128
 most_known_nbrs 1896       0.510             0.156 d0128
random_uncolored 2497       0.567             0.089 d0128
       ascending 2154       0.388             0.177 d0064
 ascending_after 1461       0.238             0.415 d0064
 most_known_nbrs 1896       0.436             0.148 d0064
random_uncolored 2497       0.486             0.113 d0064

Against `ascending_after` -- the one baseline that exploits the emission convention --
the model prefers the HEURISTIC over DD, and more so at d_model=64.


### 3c. Causal counterfactual + patching (Slurm 48583444)

One-token intervention: node `u` is forced when its known neighbours cover `q-1` colors leaving `c` free.
Recolor the neighbour that *uniquely* holds color `a` to `c`; now `a` is free instead, so DD's answer flips
`c -> a`. Exactly one token changes.

`recovery = (patched - base) / (ref - base)` on `logit_diff = logit[a] - logit[c]`. Patching the prediction
position at the final layer must give 1.0 **by construction** -- that cell is a control, asserted in
`tests/test_sudoku_circuits.py`, not a result.

In [7]:
import json
C = f"{S}/sudoku_circuits_2026-10-19/d0128_L12_v2"
circ = json.load(open(f"{C}/summary.json"))
print("clean_correct    %.3f   (the model does solve these cleanly)" % circ["clean_correct"])
print("corrupt_flipped  %.3f   vs 1/q = 0.25 chance" % circ["corrupt_flipped"])
print("logit_effect    %+.2f   (mean shift in logit_diff)" % circ["logit_effect"])
print("\n=> the model does NOT apply the forcing rule at DECISION level, but neighbour-color")
print("   information does flow substantially without dominating the argmax.")

pa = pd.read_csv(f"{C}/patching.csv")
heat = pa.pivot_table(index="layer", columns="site", values="recovery")
print("\nmean recovery by layer x site:")
print(heat.round(3).to_string())

clean_correct    0.973   (the model does solve these cleanly)
corrupt_flipped  0.267   vs 1/q = 0.25 chance
logit_effect    +12.26   (mean shift in logit_diff)

=> the model does NOT apply the forcing rule at DECISION level, but neighbour-color
   information does flow substantially without dominating the argmax.

mean recovery by layer x site:
site   edited_hint  graph_row_u  graph_row_w  predict_pos
layer                                                    
0            0.893        0.002       -0.000        0.016
1            0.864        0.004       -0.001        0.045
2            0.852        0.005       -0.003        0.067
3            0.843        0.006       -0.004        0.102
4            0.833        0.007       -0.005        0.143
5            0.818        0.006       -0.005        0.171
6            0.597        0.004        0.093        0.201
7            0.314        0.002        0.204        0.287
8            0.082        0.004        0.290        0.492
9            0.

The patching map is the most informative single result here, and it does localize a route:

| layers | where the effect lives |
|---|---|
| 0-5 | `edited_hint` holds essentially all of it (0.89 -> 0.82) |
| 6-9 | hands off; `graph_row_w` (**the recolored neighbour's** row) carries a transient 0.09 -> 0.29 |
| 8-11 | `predict_pos` takes over (0.49 -> 1.00; the 1.00 at L10-11 is the control) |

`graph_row_u` -- the row of the node *being predicted* -- sits at ~0.00 at every layer. That is the
diagnostic point: DD's forcing rule requires reading `u`'s neighbour list and joining it to the known
colors, and the effect never passes through that row. The information instead moves via the recolored
neighbour's own row, i.e. **color-to-node binding rather than a neighbour-list join**.

### Head ablation (Slurm 48585025)

The first attempt (`48583444`) returned `effect_retained == 1.0` on all 7,200 rows, which reads as "no head
matters" but actually meant *nothing was ablated*: `nn.MultiheadAttention` hands `out_proj.weight` to
`F.multi_head_attention_forward` as a plain tensor, so the `out_proj` module never runs and a
`forward_pre_hook` on it never fires. Fixed in `d3fc606` by zeroing the head's columns of `out_proj.weight`
directly, with a test that asserts the ablation moves the effect. The patching map above came from the same
run and is unaffected.

In [8]:
ab = pd.read_csv(f"{S}/sudoku_circuits_2026-10-19/d0128_L12_v3/head_ablation.csv")
per_head = ab.groupby(["layer", "head"]).effect_retained.mean().sort_values()
print("mean effect_retained across all %d (layer, head, case) rows: %.3f" % (len(ab), ab.effect_retained.mean()))
print("\nmost necessary heads (least effect retained when ablated):")
print(per_head.head(6).round(3).to_string())
print("\nNo single head is necessary -- the worst ablation keeps %.0f%% of the effect." % (100 * per_head.iloc[0]))
print("The five that matter most (L06H2, L07H1, L08H0, L09H0, L10H3) all sit in the")
print("L6-L10 handoff window that the patching map independently identifies.")

mean effect_retained across all 7200 (layer, head, case) rows: 0.986

most necessary heads (least effect retained when ablated):
layer  head
10     3       0.807
7      1       0.849
6      2       0.855
9      0       0.859
8      0       0.899
9      3       0.958

No single head is necessary -- the worst ablation keeps 81% of the effect.
The five that matter most (L06H2, L07H1, L08H0, L09H0, L10H3) all sit in the
L6-L10 handoff window that the patching map independently identifies.


### 3d. SAE features (Slurm 48583439)

The previous SAE run (`48581592`) is **not** in this table because it was uninterpretable: a fixed
`l1_coeff=0.1` gave L0 = 84..278 of 2048 with *zero* dead features while reporting FVE=0.992. The
reconstruction term is a **sum** over dimensions, so it scales with `d_model` while L1 does not, and a
coefficient calibrated at `d_model=12` is ~10x too weak at 128. FVE is not a health check; L0 and the dead
count are. `--target-l0` now searches `l1_coeff` and records `hit_target_l0`.

In [9]:
sae = pd.read_csv(f"{S}/sudoku_sae_2026-10-19_sparse/sae_summary.csv")
print(sae[["layer", "l0", "n_dead_features", "hit_target_l0", "frac_variance_explained",
           "n_interpretable", "best_frontier_f1", "family_counts"]].to_string(index=False))

 layer        l0  n_dead_features  hit_target_l0  frac_variance_explained  n_interpretable  best_frontier_f1                      family_counts
     0  1.858931             1915           True                 0.986222               24          0.074301  {"next_node": 23, "remaining": 1}
     4 16.333749                0           True                 0.954341              135          0.253800 {"next_node": 129, "remaining": 6}
     8 24.919805                0           True                 0.928754              117          0.227620 {"next_node": 110, "remaining": 7}
    12 33.710110                0           True                 0.911435               89          0.261414  {"next_node": 80, "remaining": 9}


Consistent with the probes: interpretable features are almost entirely `next_node`, there is **no**
`frontier` family at any layer, and `best_frontier_f1` tops out at 0.26. Two cautions against over-reading
this page: (i) `n_dead_features = 0` with all 2048 alive at layers 4-12 means "alive" is a weak criterion
(fires at least once in 26,448 positions), so the dictionary is sparse per-position but not well
concentrated; (ii) the 23 `next_node` features at **layer 0** cannot be computing anything -- layer 0 is the
raw token+position embedding -- so they are position features riding the same sorted-emission confound as
3b, and the deeper `next_node` counts deserve the same suspicion.

## 4. Comparison to previous best, same eval

`agg_solve_rate` on the standard removal-path eval. The scratchpad variant (Slurm `48579019`) was this
session's only new training run, and it did **not** improve on the baseline.

In [10]:
comp = pd.DataFrame([
    ("d0128 @900ep (previous best)", 0.767, "48513926_1"),
    ("d0128 @900ep + scratchpad tokens", 0.740, "48579019"),
    ("d0128 @900ep, random-hint ensemble", 0.753, "48579545"),
], columns=["run", "agg_solve_rate", "slurm"])
comp["delta_vs_best"] = (comp.agg_solve_rate - 0.767).round(4)
print(comp.to_string(index=False))
print("\nNo improvement over 0.767. The random-hint number is not comparable as a model quality")
print("figure -- it is trained and evaluated on a different hint ensemble.")

                               run  agg_solve_rate      slurm  delta_vs_best
      d0128 @900ep (previous best)           0.767 48513926_1          0.000
  d0128 @900ep + scratchpad tokens           0.740   48579019         -0.027
d0128 @900ep, random-hint ensemble           0.753   48579545         -0.014

No improvement over 0.767. The random-hint number is not comparable as a model quality
figure -- it is trained and evaluated on a different hint ensemble.


## 5. Conclusions

1. **The hint-density transition exists and is measurable on existing checkpoints.** solve_rate falls
   0.99 -> 0.44 monotonically across the removal-path ensemble, with `truncated_rate = 0.000` in every bin,
   so it is not a generation-budget artifact. It is *not* a solvability transition -- every instance is
   resolvable by construction. Whether the controlling variable is deduction depth `T` or hint density rho
   is **not resolved**: the two are collinear in this ensemble (section 2). The independent-random-hint
   ensemble has no transition at any N, which retires the earlier "scale N to ~1700 tokens" plan.
2. **The model is not running DD.** Probes (`frontier_mask` 0.67 vs `known_mask` 0.91), the head-to-head
   vote (loses to `ascending_after`), the causal counterfactual (flip rate 0.27 vs 0.25 chance) and the SAE
   (no `frontier` features) all point the same way. Best current description: **bookkeeping of which nodes
   are colored, plus the sorted-emission convention, plus sub-threshold sensitivity to neighbour colors.**
3. **That sensitivity is real and localized.** The counterfactual moves the logit gap by 12.3, routed
   `edited_hint` (L0-5) -> recolored neighbour's graph row (L6-9) -> prediction position (L8-11), and
   conspicuously *not* through the predicted node's own graph row.

**Open:** "when during training" needs the checkpoint series (Slurm `48580229` running, probes `48582169`
queued behind it); head ablation rerunning as `48585025`; and the depth-vs-density disentangling above.